In [6]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # debug chuẩn dòng lỗi (có thể tắt sau khi ổn)

import json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


In [7]:
PATH_OLD = r"C:\Users\Admin\Documents\GitHub\Thesis-Computer-Vision\recognition\weights\model_for_inference.pth"

ckpt = torch.load(PATH_OLD, map_location="cpu", weights_only=False)  # PyTorch 2.6+
print("ckpt keys:", ckpt.keys())

cfg = ckpt["config"]
NUM_CLASSES = int(cfg["num_classes"])
EMB_DIM = int(cfg["embedding_dim"])
IMG_SIZE = int(cfg["img_size"])
S = float(cfg["arcface_s"])
M = float(cfg["arcface_m"])

print("cfg:", cfg)
print("NUM_CLASSES, EMB_DIM, IMG_SIZE, S, M =", NUM_CLASSES, EMB_DIM, IMG_SIZE, S, M)

brand_to_idx = ckpt["brand_to_idx"]
idx_to_brand = ckpt["idx_to_brand"]
state_old = ckpt["model_state_dict"]
print("state keys:", len(state_old))


ckpt keys: dict_keys(['model_state_dict', 'config', 'test_accuracy', 'brand_to_idx', 'idx_to_brand'])
cfg: {'num_classes': 2984, 'embedding_dim': 512, 'img_size': 224, 'arcface_s': 64.0, 'arcface_m': 0.5}
NUM_CLASSES, EMB_DIM, IMG_SIZE, S, M = 2984 512 224 64.0 0.5
state keys: 327


In [8]:
DATA_DIR = r"C:\Users\Admin\Documents\GitHub\Thesis-Computer-Vision\data\LogoDet-3K"
ANN_PATH = os.path.join(DATA_DIR, "annotations.json")
print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("ANN exists:", os.path.exists(ANN_PATH))

with open(ANN_PATH, "r", encoding="utf-8") as f:
    ann = json.load(f)

# mapping COCO category_id -> category_name
catid_to_name = {c["id"]: c["name"] for c in ann["categories"]}

# image_id -> (file_name, split)
imgid_to_info = {im["id"]: (im["file_name"], im.get("split","train")) for im in ann["images"]}

# build samples: each annotation is a training sample (crop-less classification on full image)
# NOTE: If 1 image has multiple logos, this creates multiple entries (common for classification baseline)
samples_by_split = {"train": [], "val": [], "test": []}

missing_brand = 0
for a in ann["annotations"]:
    img_id = a["image_id"]
    cat_id = a["category_id"]
    name = catid_to_name.get(cat_id, None)
    if name is None or name not in brand_to_idx:
        missing_brand += 1
        continue

    file_name, split = imgid_to_info[img_id]
    y = int(brand_to_idx[name])

    # file path: in your dataset file_name looks like "train_084970.jpg" etc
    # You have folders train/ val/ test containing those images.
    img_path = os.path.join(DATA_DIR, split, file_name)
    if os.path.exists(img_path):
        samples_by_split[split].append((img_path, y))

print("missing_brand annotations:", missing_brand)
for sp in ["train","val","test"]:
    ys = [y for _, y in samples_by_split[sp]]
    print(sp, "num_samples:", len(ys), "| min/max:", (min(ys) if ys else None, max(ys) if ys else None))


DATA_DIR exists: True
ANN exists: True
missing_brand annotations: 0
train num_samples: 40000 | min/max: (0, 2983)
val num_samples: 5000 | min/max: (2, 2983)
test num_samples: 5000 | min/max: (1, 2983)


In [9]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import torch
torch.set_num_threads(1)

print("torch threads set")


torch threads set


In [10]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class LogoDet3KCls(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(y)

train_dataset = LogoDet3KCls(samples_by_split["train"], transform=train_tf)
val_dataset   = LogoDet3KCls(samples_by_split["val"],   transform=val_tf)

BATCH_SIZE = 32
from torch.utils.data import DataLoader

NUM_WORKERS = 0   # thử 2 trước, ok rồi tăng 4
PIN = True

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=False
)


print("train/val:", len(train_dataset), len(val_dataset))


train/val: 40000 5000


In [11]:
import time
t0 = time.time()
images, labels = next(iter(train_loader))
print("✅ first batch loaded in", round(time.time()-t0, 3), "sec", images.shape, labels.shape)
print("labels min/max:", labels.min().item(), labels.max().item())


✅ first batch loaded in 1.728 sec torch.Size([32, 3, 224, 224]) torch.Size([32])
labels min/max: 97 2950


In [12]:
def scan_labels(ds, num_classes, name="ds"):
    ys = [int(y) for _, y in ds.samples]
    mn, mx = min(ys), max(ys)
    bad = [y for y in ys if y < 0 or y >= num_classes]
    print(f"{name}: min={mn} max={mx} bad_count={len(bad)}")
    if bad:
        print("bad sample labels:", bad[:20])
    assert len(bad) == 0, "Found labels out of range!"

scan_labels(train_dataset, NUM_CLASSES, "train")
scan_labels(val_dataset,   NUM_CLASSES, "val")


train: min=0 max=2983 bad_count=0
val: min=2 max=2983 bad_count=0


In [13]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, emb, labels):
        # normalize
        emb = F.normalize(emb)
        W = F.normalize(self.weight)

        cosine = F.linear(emb, W)              # [B, C]
        theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
        target_logit = torch.cos(theta + self.m)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        output = cosine * (1 - one_hot) + target_logit * one_hot
        output *= self.s
        return output

class ResNetArcFace(nn.Module):
    def __init__(self, num_classes, emb_dim=512, s=64.0, m=0.5):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        in_feat = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.embedding = nn.Linear(in_feat, emb_dim)
        self.bn = nn.BatchNorm1d(emb_dim)
        self.dropout = nn.Dropout(0.3)
        self.arc = ArcMarginProduct(emb_dim, num_classes, s=s, m=m)

    def forward(self, x, labels):
        feat = self.backbone(x)
        emb = self.embedding(feat)
        emb = self.bn(emb)
        emb = F.relu(emb)
        emb = self.dropout(emb)
        logits = self.arc(emb, labels)
        return logits, emb

model = ResNetArcFace(NUM_CLASSES, EMB_DIM, S, M).to(device)

# load old weights into backbone/embedding/bn (ignore arc head)
state = {k.replace("module.", ""): v for k, v in state_old.items()}
# drop possible classifier keys from old
for k in list(state.keys()):
    if k.startswith("classifier."):
        state.pop(k)

missing, unexpected = model.load_state_dict(state, strict=False)
print("missing:", missing)
print("unexpected:", unexpected)

# quick sanity
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.long().to(device)
logits, emb = model(images, labels)
print("logits:", logits.shape, "emb:", emb.shape, "labels min/max:", labels.min().item(), labels.max().item())


missing: ['arc.weight']
unexpected: []
logits: torch.Size([32, 2984]) emb: torch.Size([32, 512]) labels min/max: 106 2892


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)


In [ ]:
def run_one_epoch_safe(model, loader, optimizer, criterion, device, train=True, epoch=1, epochs=1, num_classes=2984):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, total=len(loader), desc=("Train" if train else "Val")+f" {epoch}/{epochs}")

    for bi, (images, labels) in enumerate(pbar, start=1):
        # CPU check before GPU
        labels = labels.long()
        mn, mx = int(labels.min().item()), int(labels.max().item())
        if mn < 0 or mx >= num_classes:
            print(f"\n❌ BAD LABEL batch={bi}: min={mn} max={mx} num_classes={num_classes}")
            raise ValueError("Label out of range")

        images = images.to(device)
        labels = labels.to(device)

        if train:
            logits, _ = model(images, labels)
            if logits.size(1) != num_classes:
                raise ValueError(f"logits dim mismatch: {logits.size(1)} vs {num_classes}")

            loss = criterion(logits, labels)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                logits, _ = model(images, labels)
                if logits.size(1) != num_classes:
                    raise ValueError(f"logits dim mismatch: {logits.size(1)} vs {num_classes}")
                loss = criterion(logits, labels)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct += (logits.argmax(1) == labels).sum().item()
        total += bs
        pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.4f}")

    return total_loss/total, correct/total


import os
SAVE_DIR = "./recognition/weights"
os.makedirs(SAVE_DIR, exist_ok=True)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = -1.0

EPOCHS = 2

for epoch in range(1, EPOCHS+1):
    t0 = time.time()

    train_loss, train_acc = run_one_epoch_safe(model, train_loader, optimizer, criterion, device, True,  epoch, EPOCHS, NUM_CLASSES)
    val_loss,   val_acc   = run_one_epoch_safe(model, val_loader,   optimizer, criterion, device, False, epoch, EPOCHS, NUM_CLASSES)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"[Epoch {epoch:02d}/{EPOCHS}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | {time.time()-t0:.1f}s")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "config": cfg,
        "brand_to_idx": brand_to_idx,
        "idx_to_brand": idx_to_brand,
    }, os.path.join(SAVE_DIR, "arcface_last.pth"))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "best_val_acc": best_val_acc,
            "config": cfg,
            "brand_to_idx": brand_to_idx,
            "idx_to_brand": idx_to_brand,
        }, os.path.join(SAVE_DIR, "arcface_best.pth"))
        print(f"✅ Saved BEST: val_acc={best_val_acc:.4f}")


Train 1/2:  20%|█▉        | 248/1250 [45:18<3:11:29, 11.47s/it, acc=0.0000, loss=41.2651]

In [ ]:
import matplotlib.pyplot as plt

ep = range(1, len(history["train_loss"]) + 1)

plt.figure()
plt.plot(ep, history["train_loss"], label="train_loss")
plt.plot(ep, history["val_loss"], label="val_loss")
plt.legend(); plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss"); plt.show()

plt.figure()
plt.plot(ep, history["train_acc"], label="train_acc")
plt.plot(ep, history["val_acc"], label="val_acc")
plt.legend(); plt.xlabel("epoch"); plt.ylabel("acc"); plt.title("Accuracy"); plt.show()
